In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)

freq   = pd.read_csv(PROC / "frequency_data.csv").rename(columns={'class': 'class_year'})
claims = pd.read_csv(PROC / "claims.csv").rename(columns={'class': 'class_year'})

freq_coefs = pd.read_csv(PROC / "freq_coefficients.csv")
sev_coefs  = pd.read_csv(PROC / "sev_coefficients.csv")
coefs      = pd.concat([freq_coefs, sev_coefs], ignore_index=True)

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

RATING_VARS = ['class_year', 'study', 'gpa', 'greek', 'gender',
               'off_campus', 'distance_to_campus', 'sprinklered']

CAT_VARS = ['class_year', 'study', 'greek', 'gender', 'off_campus']

coefs

,coverage,term,coef,pval,se,component
0,Personal Property,Intercept,-1.642661,5.366536e-38,0.127472,frequency
1,Personal Property,greek = Non-greek,-0.796369,1.710577e-26,0.074764,frequency
2,Personal Property,sprinklered = True,0.037408,6.345637e-01,0.078701,frequency
3,Personal Property,gpa,-0.051545,2.190878e-01,0.041942,frequency
4,Additional Living Expense,Intercept,-2.304637,1.937680e-99,0.108880,frequency
5,Additional Living Expense,greek = Non-greek,-0.751963,1.860888e-13,0.102191,frequency
6,Additional Living Expense,sprinklered = True,-0.103147,3.207081e-01,0.103873,frequency
7,Guest Medical,Intercept,-3.184421,6.085054e-32,0.270725,frequency
8,Guest Medical,greek = Non-greek,-0.592791,1.930457e-04,0.159013,frequency
9,Guest Medical,study = Humanities,0.030186,8.819078e-01,0.203205,frequency


Now to make the basic prediction function for expected policy cost

In [2]:
def linear_predictor(risk, coverage, component):
    d   = coefs[(coefs.coverage == coverage) & (coefs.component == component)]
    eta = 0.0
    for term, b in zip(d.term, d.coef):
        if term == 'Intercept':
            eta += b
        elif ' = ' in term:
            var, level = term.split(' = ', 1)
            if str(risk[var]) == level:
                eta += b
        else:
            eta += b * float(risk[term])
    return eta


def expected_cost(risk, detail=False):
    rows = []
    for c in COVERAGES:
        f = np.exp(linear_predictor(risk, c, 'frequency'))
        s = np.exp(linear_predictor(risk, c, 'severity'))
        rows.append({'coverage': c, 'frequency': f,
                     'severity': s, 'pure_premium': f * s})

    out = pd.DataFrame(rows)
    return out if detail else out.pure_premium.sum()

Now to test the function

In [9]:
risk = {
    'class_year':         'Freshman',
    'study':              'Science',
    'gpa':                3.5,
    'greek':              'Greek',
    'off_campus':         'Off campus',
    'distance_to_campus': 1.2,
    'sprinklered':        False,
}

print(round(expected_cost(risk), 2))
expected_cost(risk, detail=True).round(4)

2236.26


,coverage,frequency,severity,pure_premium
0,Personal Property,0.1615,2617.6158,422.8214
1,Additional Living Expense,0.0998,9486.8109,946.7366
2,Guest Medical,0.0333,21795.7110,725.3091
3,Liability,0.0182,7756.9635,141.3918


In [11]:
risk = {
    'class_year':         'Freshman',
    'study':              'Science',
    'gpa':                3.5,
    'greek':              'Greek',
    'off_campus':         'Off campus',
    'distance_to_campus': 1.2,
    'sprinklered':        False,
}

print(round(expected_cost(risk), 2))
expected_cost(risk, detail=True).round(4)

2236.26


,coverage,frequency,severity,pure_premium
0,Personal Property,0.1615,2617.6158,422.8214
1,Additional Living Expense,0.0998,9486.8109,946.7366
2,Guest Medical,0.0333,21795.7110,725.3091
3,Liability,0.0182,7756.9635,141.3918
